# HELM — analyser votre commentaire
Saisissez un texte français puis obtenez sa prédiction et ses explications selon votre profil. Démonstration fonctionnelle ; aucune préférence humaine apprise.

## 1. Récupérer HELM depuis GitHub
Sources : `4fee0dc34b891dd98766cb4191ee1613d53739ee`. Dépôt privé, lien temporaire masqué. Calculs exécutés dans Colab.

In [ ]:
import hashlib
import io
import sys
import urllib.request
import zipfile
from pathlib import Path
from getpass import getpass

REVISION = "4fee0dc34b891dd98766cb4191ee1613d53739ee"
if globals().get("ARCHIVE_REVISION") != REVISION:
    archive_url = getpass("Lien GitHub temporaire de la révision : ")
    assert archive_url.startswith("https://codeload.github.com/KossiFO/helm-xai-these/")
    archive = urllib.request.urlopen(archive_url, timeout=120).read()
    del archive_url
    ARCHIVE_REVISION = REVISION
source_dir = Path("/content/helm_github_source")
source_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
    root_name = zipped.namelist()[0].split("/")[0]
    archive_revision = root_name.rsplit("-", 1)[-1]
    assert len(archive_revision) >= 7 and REVISION.startswith(archive_revision), "Révision inattendue"
    assert all((source_dir / n).resolve().is_relative_to(source_dir.resolve()) for n in zipped.namelist())
    zipped.extractall(source_dir)
SOURCE = source_dir / root_name
for module_name in list(sys.modules):
    if module_name == "helm_colab" or module_name.startswith("helm_colab."):
        del sys.modules[module_name]
sys.path.insert(0, str(SOURCE / "examples/colab"))
print("Codes récupérés depuis GitHub :", REVISION)
print("Archive SHA256 :", hashlib.sha256(archive).hexdigest())
print("Sources consultables dans le panneau Fichiers :", SOURCE)


## 2. Installer HELM dans Colab
Environnement Python 3.12 séparé ; GPU si disponible. Le premier téléchargement du modèle prend quelques minutes.

In [ ]:
from helm_colab.toxicity_session import ToxicityExperiment
toxicite = ToxicityExperiment()
toxicite.install()

## 3. Saisir votre commentaire
Lancez cette cellule une fois, puis utilisez le champ. **Prédire** donne le score ; **Expliquer** calcule aussi les cinq méthodes. Choisissez un profil avant ou après le calcul ; les résultats sont réutilisés. Gardez la session Colab connectée.
Limite : 4 000 caractères et 512 tokens, sans troncature silencieuse.

In [ ]:
saisie = toxicite.interactive()